# 02 — Vector Store RAG: Python Tips Knowledge Base

Move from manual numpy search to a proper **FAISS vector store**.

### New concepts
| Concept | Description |
|---------|-------------|
| FAISS | Facebook AI Similarity Search — fast approximate nearest-neighbour index |
| Persistence | Save the index to disk and reload it without re-embedding |
| Semantic vs keyword | Why `"how to remember values"` finds `memoisation` tips |

### No LLM or API key needed in this notebook.

## 1. Install Dependencies

In [ ]:
# !pip install langchain langchain-community sentence-transformers faiss-cpu numpy

## 2. Imports

In [ ]:
import os, pickle
import numpy as np
import faiss
from pathlib import Path
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

print("Imports OK")

## 3. Load the Knowledge Base

In [ ]:
# Read our Python tips text file
data_path = Path("../data/python_tips.txt")
text = data_path.read_text(encoding="utf-8")

# Wrap in a single Document
doc = Document(page_content=text, metadata={"source": "python_tips.txt", "topic": "python"})

print(f"Loaded document: {len(text)} characters")
print(f"Preview: {text[:200]}...")

## 4. Chunk the Document

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_documents([doc])
chunk_texts = [c.page_content for c in chunks]

print(f"Split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i}: {chunk.page_content[:100]}...")

## 5. Generate Embeddings

In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
dim   = model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {dim}")

print("\nEmbedding chunks...")
embeddings = model.encode(chunk_texts, show_progress_bar=True).astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

## 6. Build the FAISS Index

FAISS stores all vectors in an efficient index structure.
`IndexFlatL2` does exact search using Euclidean distance — simple and accurate for small datasets.

In [ ]:
# Build index
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"FAISS index built. Total vectors: {index.ntotal}")

# ── Save to disk ────────────────────────────────────────────────────────
store_dir = Path("faiss_python_tips")
store_dir.mkdir(exist_ok=True)

faiss.write_index(index, str(store_dir / "index.faiss"))
with open(store_dir / "chunks.pkl", "wb") as f:
    pickle.dump(chunk_texts, f)

print(f"Saved index + chunks to {store_dir}/")

## 7. Load from Disk and Search

In [ ]:
# ── Load back ────────────────────────────────────────────────────────
loaded_index = faiss.read_index(str(store_dir / "index.faiss"))
with open(store_dir / "chunks.pkl", "rb") as f:
    loaded_chunks = pickle.load(f)

print(f"Loaded {loaded_index.ntotal} vectors and {len(loaded_chunks)} chunks")

# ── Semantic search function ──────────────────────────────────────────
def semantic_search(query: str, top_k: int = 3) -> list:
    q_emb = model.encode([query]).astype("float32")
    distances, indices = loaded_index.search(q_emb, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "chunk": loaded_chunks[idx],
            "distance": float(dist),
            "score": 1 / (1 + float(dist))   # convert L2 distance to 0-1 score
        })
    return results

## 8. Semantic Search in Action

In [ ]:
queries = [
    "how to make code run faster",
    "remembering expensive function results",
    "reading and writing files safely",
    "create lists in one line",
]

for query in queries:
    print(f"\n{'─'*55}")
    print(f"  Query: {query}")
    print(f"{'─'*55}")
    results = semantic_search(query, top_k=2)
    for r in results:
        print(f"  Score {r['score']:.3f} | {r['chunk'][:120]}...")

## 9. Semantic vs Keyword Search

See why semantic search is more powerful than simple keyword matching.

In [ ]:
def keyword_search(query: str, texts: list, top_k: int = 3) -> list:
    """Simple keyword matching — counts how many query words appear in each chunk."""
    query_words = set(query.lower().split())
    scored = []
    for text in texts:
        text_words = set(text.lower().split())
        overlap = len(query_words & text_words)
        scored.append((text, overlap))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [t for t, _ in scored[:top_k]]

# ── Compare both approaches on a vague query ─────────────────────────
vague_query = "cache function output"

print("SEMANTIC SEARCH results:")
for r in semantic_search(vague_query, top_k=2):
    print(f"  [{r['score']:.3f}] {r['chunk'][:130]}...")

print("\nKEYWORD SEARCH results:")
for text in keyword_search(vague_query, chunk_texts, top_k=2):
    print(f"  {text[:130]}...")

## 10. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| FAISS index | Stores embeddings for fast nearest-neighbour search |
| `IndexFlatL2` | Exact search via Euclidean distance |
| Persistence | `write_index` / `read_index` saves/loads without re-embedding |
| L2 → score | `1 / (1 + distance)` converts distance to 0–1 similarity |
| Semantic > keyword | Semantic search understands meaning, not just word overlap |

**Next notebook →** connect the retriever to a Groq LLM for a complete Q&A pipeline.